# Capstone

This capstone project will be building a service to identify what sport is portrayed by a picture.  This service will be hosted in a kubernetes cluster.

In [15]:
import numpy as np
import matplotlib.pyplot as plt
import requests
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.applications.xception import decode_predictions

%matplotlib inline

In [2]:
train_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_ds = train_gen.flow_from_directory(
    './data/train',
    target_size=(224, 224),
    batch_size=32
)

Found 13492 images belonging to 100 classes.


In [3]:
val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

val_ds = val_gen.flow_from_directory(
    './data/valid',
    target_size=(224, 224),
    batch_size=32,
    shuffle=False
)

Found 500 images belonging to 100 classes.


In [4]:
def make_model(base_model, learning_rate, size_inner, droprate):
    inputs = keras.Input(shape=(224, 224, 3))
    base = base_model(inputs, training=False)
    vectors = keras.layers.GlobalAveragePooling2D()(base)

    inner = keras.layers.Dense(size_inner, activation='relu')(vectors)
    drop = keras.layers.Dropout(droprate)(inner)
    
    outputs = keras.layers.Dense(100)(drop)
    model = keras.Model(inputs, outputs)

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    loss = keras.losses.CategoricalCrossentropy(from_logits=True)

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['accuracy']
    )

    return model

In [6]:
def get_callbacks():
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]

base_model = Xception(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

best_val_acc = -1
best_lr = 0.001
best_si = 50
best_dr = 0.5

for lr in [0.001, 0.01, 0.1]:
    model = make_model(base_model, lr, best_si, best_dr)
    history = model.fit(
        train_ds,
        epochs=3,
        callbacks=get_callbacks(),
        validation_data=val_ds
    )
    
    max_val_acc = max(history.history["val_accuracy"])
    if max_val_acc > best_val_acc:
        best_val_acc = max_val_acc
        best_lr = lr


Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 749s 2s/step - accuracy: 0.2510 - loss: 3.1689 - val_accuracy: 0.7020 - val_loss: 1.4750
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 764s 2s/step - accuracy: 0.4676 - loss: 1.9429 - val_accuracy: 0.7640 - val_loss: 1.0060
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 660s 2s/step - accuracy: 0.5351 - loss: 1.6254 - val_accuracy: 0.8060 - val_loss: 0.7843
Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 672s 2s/step - accuracy: 0.0704 - loss: 4.0996 - val_accuracy: 0.1740 - val_loss: 3.2498
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 685s 2s/step - accuracy: 0.0919 - loss: 3.8598 - val_accuracy: 0.1840 - val_loss: 3.0665
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 676s 2s/step - accuracy: 0.0986 - loss: 3.8074 - val_accuracy: 0.2180 - val_loss: 2.9626
Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 676s 2s/step - accuracy: 0.0136 - loss: 5.0051 - val_accuracy: 0.0100 - val_loss: 4.6683
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 714s 2s/step - accuracy: 0.0118 - loss: 4.6410 - val_accuracy: 0.

In [7]:
best_lr

0.001

In [8]:
# I'm not resetting best_val_acc,
# so we've already tested with an si of 50 in the previous loop
for si in [10, 100]:
    model = make_model(base_model, best_lr, si, best_dr)
    history = model.fit(
        train_ds,
        epochs=3,
        callbacks=get_callbacks(),
        validation_data=val_ds
    )
    
    max_val_acc = max(history.history["val_accuracy"])
    if max_val_acc > best_val_acc:
        best_val_acc = max_val_acc
        best_si = si

Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 681s 2s/step - accuracy: 0.0474 - loss: 4.2272 - val_accuracy: 0.2360 - val_loss: 3.5684
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 693s 2s/step - accuracy: 0.0878 - loss: 3.7725 - val_accuracy: 0.3120 - val_loss: 3.0321
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 699s 2s/step - accuracy: 0.0954 - loss: 3.6220 - val_accuracy: 0.3720 - val_loss: 2.8305
Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 701s 2s/step - accuracy: 0.3873 - loss: 2.5405 - val_accuracy: 0.7640 - val_loss: 0.9595
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 702s 2s/step - accuracy: 0.6269 - loss: 1.3344 - val_accuracy: 0.8280 - val_loss: 0.6161
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 704s 2s/step - accuracy: 0.6938 - loss: 1.0644 - val_accuracy: 0.8560 - val_loss: 0.5304


In [9]:
best_si

100

In [10]:
# I'm not resetting best_val_acc,
# so we've already tested with a dr of 0.5 in the previous two loops
for dr in [0.0, 0.8]:
    model = make_model(base_model, best_lr, best_si, dr)
    history = model.fit(
        train_ds,
        epochs=3,
        callbacks=get_callbacks(),
        validation_data=val_ds
    )
    
    max_val_acc = max(history.history["val_accuracy"])
    if max_val_acc > best_val_acc:
        best_val_acc = max_val_acc
        best_dr = dr


Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 688s 2s/step - accuracy: 0.6247 - loss: 1.5326 - val_accuracy: 0.8240 - val_loss: 0.6169
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 698s 2s/step - accuracy: 0.8467 - loss: 0.5430 - val_accuracy: 0.8580 - val_loss: 0.4703
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 699s 2s/step - accuracy: 0.8933 - loss: 0.3672 - val_accuracy: 0.8860 - val_loss: 0.3835
Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 845s 2s/step - accuracy: 0.1014 - loss: 3.9708 - val_accuracy: 0.5380 - val_loss: 2.5249
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 815s 2s/step - accuracy: 0.2066 - loss: 3.1299 - val_accuracy: 0.6800 - val_loss: 1.7014
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 773s 2s/step - accuracy: 0.2528 - loss: 2.7833 - val_accuracy: 0.7200 - val_loss: 1.3797


In [11]:
print(f"Best\n\tLearning Rate: {best_lr},\n\tSize Inner: {best_si},\n\tDroprate: {best_dr}")

Best
	Learning Rate: 0.001,
	Size Inner: 100,
	Droprate: 0.0


# Setup Deployment Environment

In [14]:
# I will only run this once, since the lockfile will be committed
#!uv init
#!uv add fastapi==0.121.1 uvicorn==0.38.0 tensorflow==2.20.0 pillow==10.4.0 python-multipart==0.0.9

In [19]:
url = "http://capstone-service:9697/predict"

with open("./data/test/giant slalom/3.jpg", "rb") as f:
    files = {"file": f}
    response = requests.post(url, files=files)

print(response.status_code)
response.json()

200


{'predictions': [{'class': 'giant slalom', 'confidence': 5.526092052459717},
  {'class': 'ski jumping', 'confidence': 3.2777397632598877},
  {'class': 'canoe slamon', 'confidence': 1.66562819480896}]}